# ED.03.03 - DataFrames en PySpark

Gustavo Adalid Catalán Carmen IDIA 224

Ejercicios prácticos 1-8 y proyecto de análisis de ventas

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, desc, avg, max, min, sum, count, rank, trim, concat_ws, when, to_date, year
)

spark = (
    SparkSession.builder.appName("ED03-03-DataFrames")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Ejercicio 1: Creación y Exploración

In [ ]:
datos = [
    ("Ana", 28, "Ventas", 35000),
    ("Juan", 35, "IT", 45000),
    ("María", 42, "IT", 55000),
    ("Pedro", 31, "Ventas", 38000),
    ("Laura", 29, "Marketing", 40000),
]
df = spark.createDataFrame(datos, ["nombre", "edad", "departamento", "salario"])
df.printSchema()
df.show(3)
print(f"Total de empleados: {df.count()}")

## Ejercicio 2: Filtrado

Empleados de IT con salario > 40000

In [ ]:
df.filter((col("departamento") == "IT") & (col("salario") > 40000)).show()

Empleados de Ventas o Marketing

In [ ]:
df.filter((col("departamento") == "Ventas") | (col("departamento") == "Marketing")).show()

Empleados que no son de IT

In [ ]:
df.filter(~(col("departamento") == "IT")).show()

Empleados entre 30 y 40 años

In [ ]:
df.filter((col("edad") >= 30) & (col("edad") <= 40)).show()

## Ejercicio 3: Manipulación de Columnas

In [ ]:
df3 = df.withColumn("salario_en_5_años", col("salario") + (col("salario") * 0.05 * 5))
df3 = df3.withColumnRenamed("nombre", "empleado")
df3.show()

## Ejercicio 4: Agregaciones

Salario promedio por departamento

In [ ]:
df.groupBy("departamento").agg(avg("salario").alias("salario_promedio")).show()

Salario máximo y mínimo por departamento

In [ ]:
df.groupBy("departamento").agg(
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo"),
).show()

Empleados por departamento

In [ ]:
df.groupBy("departamento").agg(count("*").alias("total_empleados")).show()

Gasto total en salarios por departamento, de mayor a menor

In [ ]:
df.groupBy("departamento").agg(sum("salario").alias("gasto_total")).orderBy(desc("gasto_total")).show()

## Ejercicio 5: Window Functions

In [ ]:
ventas = spark.createDataFrame(
    [("Enero", "ProdA", 100), ("Enero", "ProdB", 150), ("Febrero", "ProdA", 120)],
    ["mes", "producto", "ventas"],
)
ventana = Window.partitionBy("mes").orderBy(col("ventas").desc())
ventas.withColumn("ranking", rank().over(ventana)).show()

## Ejercicio 6: Joins

In [ ]:
personas = spark.createDataFrame([("Ana", 1), ("Luis", 2), ("Marta", 3)], ["Nombre", "DepID"])
departamentos = spark.createDataFrame([(1, "IT"), (2, "HR")], ["DepID", "NombreDep"])
personas.join(departamentos, "DepID", "left").select("Nombre", "DepID", "NombreDep").show()

**¿Qué pasa con Marta?** Su `DepID` es 3 y no existe en el DataFrame de departamentos. El left join la conserva, pero `NombreDep` queda en `null`.

## Ejercicio 7: SQL Puro

In [ ]:
df.createOrReplaceTempView("empleados")
spark.sql("""
    SELECT *
    FROM empleados
    WHERE departamento = 'IT' AND salario > 40000
""").show()

## Ejercicio 8: Funciones de String

In [ ]:
nombres = spark.createDataFrame([(" Juan ", "Garcia"), ("Ana", " Pérez ")], ["nombre", "apellido"])
limpio = nombres.withColumn("nombre", trim(col("nombre"))).withColumn("apellido", trim(col("apellido")))
limpio.withColumn("Nombre_Completo", concat_ws(" ", col("nombre"), col("apellido"))).show()

# Proyecto: Análisis de Ventas

Objetivo: analizar el rendimiento de ventas por país con `pdi_sales.csv`.

### Descarga del dataset

In [ ]:
!mkdir -p DATOS
!wget -q -nc "https://raw.githubusercontent.com/josepgarcia/datos/main/pdi_sales.csv" -P ./DATOS

### 1. Cargar el CSV

El separador es `;`. Las columnas `Zip` y `Country` traen espacios al final, por eso se les aplica `trim`.

In [ ]:
ventas_df = spark.read.csv("DATOS/pdi_sales.csv", header=True, inferSchema=True, sep=";")
ventas_df = ventas_df.withColumn("Zip", trim(col("Zip"))).withColumn("Country", trim(col("Country")))

### 2. Esquema y número de filas

In [ ]:
ventas_df.printSchema()
print(f"Total de filas: {ventas_df.count()}")

### 3. Total de ingresos por país, de mayor a menor

In [ ]:
ventas_df.groupBy("Country").agg(sum("Revenue").alias("Ingresos_Totales")).orderBy(desc("Ingresos_Totales")).show()

### 4. Ventas de Alemania con más de 1 unidad

In [ ]:
alemania = ventas_df.filter((col("Country") == "Germany") & (col("Units") > 1))
alemania.show(10)
print(f"Ventas encontradas: {alemania.count()}")

### 5. Código postal con más ventas en Alemania

In [ ]:
(
    ventas_df.filter(col("Country") == "Germany")
    .groupBy("Zip")
    .agg(count("*").alias("Num_Ventas"))
    .orderBy(desc("Num_Ventas"))
    .show(5)
)

### 6. Clasificar la venta: Alta si Revenue > 500, Normal si no

In [ ]:
clasificado = ventas_df.withColumn("Clasificacion", when(col("Revenue") > 500, "Alta").otherwise("Normal"))
clasificado.show(10)
clasificado.groupBy("Clasificacion").count().show()

### 7. (Bonus) Media de ingresos por año

In [ ]:
por_anio = ventas_df.withColumn("Anio", year(to_date(col("Date"), "M/d/yyyy")))
por_anio.groupBy("Anio").agg(avg("Revenue").alias("Media_Ingresos")).orderBy("Anio").show(30)

In [ ]:
spark.stop()